# Tugas 6 - UTS | Klasifikasi Berita - naïve bayes & SVM

In [5]:
# Import library
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("uts_berita.csv")  # ganti nama sesuai file Ibu
df.head()


,No,judul,berita,tanggal,kategori,link
0,1,Airlangga Harap Kenaikan UMP Tingkatkan Daya B...,Menteri Koordinator (Menko) Bidang Perekonomia...,"Minggu, 01 Des 2024 23:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...
1,2,PT SIER Beri Penghargaan untuk 50 Tenant Terba...,"Dalam rangka memeriahkan hari jadi ke-50, PT S...","Minggu, 01 Des 2024 20:45 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...
2,3,Prabowo Bakal Bentuk Kementerian Penerimaan Ne...,Wacana Presiden Prabowo Subianto akan membentu...,"Minggu, 01 Des 2024 19:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
3,4,Sinergi Kemenag & BPJS Ketenagakerjaan Lindung...,BPJS Ketenagakerjaan dan Kementerian Agama (Ke...,"Minggu, 01 Des 2024 19:03 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
4,5,Pemerintah Segera Bentuk Satgas PHK Usai Tetap...,Pemerintah akan segera membentuk Satuan Tugas ...,"Minggu, 01 Des 2024 19:00 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('indonesian'))

def clean_text(text):
    text = re.sub(r'http\S+', '', text)           # hapus link
    text = re.sub(r'[^a-zA-Z\s]', '', text)       # hapus karakter non huruf
    text = text.lower()                           # huruf kecil semua
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_berita'] = df['berita'].apply(clean_text)
df.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\endyzan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\endyzan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,No,judul,berita,tanggal,kategori,link,clean_berita
0,1,Airlangga Harap Kenaikan UMP Tingkatkan Daya B...,Menteri Koordinator (Menko) Bidang Perekonomia...,"Minggu, 01 Des 2024 23:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...,menteri koordinator menko bidang perekonomian ...
1,2,PT SIER Beri Penghargaan untuk 50 Tenant Terba...,"Dalam rangka memeriahkan hari jadi ke-50, PT S...","Minggu, 01 Des 2024 20:45 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...,rangka memeriahkan pt surabaya industrial esta...
2,3,Prabowo Bakal Bentuk Kementerian Penerimaan Ne...,Wacana Presiden Prabowo Subianto akan membentu...,"Minggu, 01 Des 2024 19:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...,wacana presiden prabowo subianto membentuk mem...
3,4,Sinergi Kemenag & BPJS Ketenagakerjaan Lindung...,BPJS Ketenagakerjaan dan Kementerian Agama (Ke...,"Minggu, 01 Des 2024 19:03 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...,bpjs ketenagakerjaan kementerian agama kemenag...
4,5,Pemerintah Segera Bentuk Satgas PHK Usai Tetap...,Pemerintah akan segera membentuk Satuan Tugas ...,"Minggu, 01 Des 2024 19:00 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...,pemerintah membentuk satuan tugas pemutusan hu...


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Ubah teks ke representasi BoW
vectorizer = CountVectorizer(max_df=0.95, min_df=2)
X = vectorizer.fit_transform(df['clean_berita'])

# Tentukan jumlah topik (misal 5 topik)
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda_features = lda.fit_transform(X)

print("Shape hasil ekstraksi topik:", lda_features.shape)


Shape hasil ekstraksi topik: (1500, 5)


In [8]:
from sklearn.model_selection import train_test_split

y = df['kategori']

X_train, X_test, y_train, y_test = train_test_split(lda_features, y, test_size=0.2, random_state=42, stratify=y)


In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Model Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Model SVM
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# Evaluasi
print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))
print("Akurasi:", accuracy_score(y_test, y_pred_nb))

print("\n=== SVM ===")
print(classification_report(y_test, y_pred_svm))
print("Akurasi:", accuracy_score(y_test, y_pred_svm))


=== Naive Bayes ===
               precision    recall  f1-score   support

      Ekonomi       0.63      0.53      0.58        75
Internasional       0.47      0.75      0.58        75
     Nasional       0.36      0.17      0.23        75
     Olahraga       0.88      0.96      0.92        75

     accuracy                           0.60       300
    macro avg       0.59      0.60      0.58       300
 weighted avg       0.59      0.60      0.58       300

Akurasi: 0.6033333333333334

=== SVM ===
               precision    recall  f1-score   support

      Ekonomi       0.65      0.53      0.58        75
Internasional       0.42      0.79      0.55        75
     Nasional       0.58      0.09      0.16        75
     Olahraga       0.84      0.97      0.90        75

     accuracy                           0.60       300
    macro avg       0.62      0.60      0.55       300
 weighted avg       0.62      0.60      0.55       300

Akurasi: 0.5966666666666667
